# Beyond BPE: Distilling a Byte Student

Allow 90-120 minutes plus the notebook experiment. Prerequisites: byte/BPE
tokenization, stable softmax, conditional probability, causal attention, and
teacher-student training. Refresh **Bytes, Tokens, and Boundaries** and
**Distillation Across Tokenizations** in the interactive track first.

By the end, you should be able to convert a teacher distribution, identify an
approximation at token boundaries, verify exact path probabilities, and train
a small student without confusing a correct target with a successful model.

## 1. Start with the mismatch

A token teacher may choose among `a`, `ab`, `ac`, and `b`. A byte student chooses
among byte values 0-255. The teacher's second logit is not the student's second
logit: their output coordinates mean different things. Distillation requires
targets in the student's output space before applying a loss.

A byte is not a character. `"e".encode("utf-8")` has one byte, while
`"\u00e9".encode("utf-8")` has two. A tokenizer entry can even be only a
fragment of a multibyte encoding. Operate on `bytes`; do not decode each token
separately, drop invalid fragments, or count characters instead of bytes.

For the exercise, vocabulary entries are unique, nonempty byte strings. We do
not model document special tokens. EOT is symbol **256**, outside the byte
alphabet, and means the end of a teacher token. EOS would mean something else.
Teacher boundaries remain part of the augmented representation.

## 2. Work the conversion by hand

Our example teacher has these probabilities:

| Token | Probability |
|---|---:|
| `a` | 0.10 |
| `ab` | 0.40 |
| `ac` | 0.30 |
| `b` | 0.20 |

Before any byte is observed, next-byte probability is `P(a)=0.8`, `P(b)=0.2`.
Now condition on prefix `a`. The matching teacher mass is 0.8, including the
complete token `a`, not just longer tokens.

**Discarding completed tokens:** only `ab` and `ac` remain. Normalize mass 0.7:
`P(b)=4/7`, `P(c)=3/7`. This distribution sums to one but has dropped a branch.
To follow that missing branch as raw bytes, we would need the teacher's next
token distribution after the completed token, which is not in this logit row.

**Keeping an explicit boundary:** assign the complete token's mass to EOT:
`P(b)=0.5`, `P(c)=0.375`, `P(EOT)=0.125`. Now the total is still one, but no
matching branch vanished. At prefix `ab`, EOT has probability one.

The complete augmented path for token `ab` has probability
`P(a) * P(b | a) * P(EOT | ab) = 0.8 * 0.5 * 1 = 0.4`.
For token `a`, the path is `0.8 * 0.125 = 0.1`. Omitting the terminating EOT
would incorrectly report 0.8 for that shorter token.

This telescoping product is the central invariant. Repeat it for every token,
not only the token chosen by the teacher. It proves conversion on the declared
unique-token vocabulary; it does not prove a learned student matches it.

## 3. Implement a stable conditional target

`byte_targets(logits, vocabulary, prefix, use_eot)` does four jobs:

1. Find byte strings starting with `prefix`.
2. Determine the next symbol: the next byte, or EOT for a completed match.
3. Normalize eligible logits using a subtracted maximum.
4. Sum the normalized mass of entries that share that next symbol.

Why normalize **after** selecting? Consider logits `[10000, 0, 0]` for
`[a, bx, by]`, conditioned on `b`. A global softmax may round both eligible
probabilities to zero. Conditional logits `[0,0]` instead give `x=y=0.5`.

`-inf` denotes zero teacher mass and is allowed. Reject NaN, positive infinity,
bad shapes, duplicate/empty tokens, and unsupported prefixes. Without EOT,
prefix `ab` in our example has no continuation; returning a uniform byte
distribution would invent knowledge. Raise `ValueError` instead.

The reference uses a linear scan and `bincount` for clarity. A large production
vocabulary benefits from prefix indexing and batched conversion. This exercise
does not claim that scanning every token for every position is scalable.

## 4. Train with soft distributions

For teacher target p and student distribution q at one position:

```text
KL(p || q) = sum_b p[b] * (log(p[b]) - log(q[b]))
```

Use temperature 1 here. Compute `log(q)` directly with stable log-softmax.
Zero p terms contribute zero; `0 * log(0)` must not create NaNs. Average over
valid positions, not over vocabulary coordinates. Padding does not contribute.
Predicted EOT symbols do contribute in the augmented representation.

The teacher entropy term is constant with respect to student weights. Thus
soft cross-entropy and forward KL have the same student gradient, but different
reported loss values. Do not compare those raw losses as if they were one metric.

`distillation_kl` grades the numerical objective. The notebook uses PyTorch
autograd to update real parameters and evaluates a separate held-out split.

## 5. Run the controlled notebook

The notebook contains this lesson, starter, exact grader, worked reference,
and a complete CPU experiment. Implement the starter before running its grader;
the untouched starter must fail. The reference cell deliberately replaces your
functions only when run. Compare explanations as well as outputs.

The experiment trains a small token teacher on synthetic segmented strings,
then trains four smaller student arms for each of two initialization seeds:

| Representation | Objective |
|---|---|
| Bytes | Supervised cross-entropy |
| Bytes | Approximate converted-target distillation |
| Bytes + EOT | Supervised cross-entropy |
| Bytes + EOT | Boundary-preserving distillation |

The synthetic segmentation is declared, not learned BPE. Train and validation
sets have disjoint raw byte strings, although they share a simple generating
rule. That is a narrow generalization test, not a natural-language benchmark.
Both supervised arms matter: they separate adding boundary information from
adding soft teacher targets. The teacher is fixed across student arms.

The schedule uses the same raw training examples in the same order. EOT adds
positions, so this is **fixed data, not fixed compute**. Record raw content bytes,
prediction units, parameter counts, training time, and the labeled `6NU` work
proxy. This proxy omits important attention and vocabulary effects and is not a
hardware FLOP measurement. Teacher training and target conversion are separate
costs, not free operations hidden in the student timing.

For an extension, hold an estimated work budget fixed by reducing the EOT arm's
steps. This changes data exposure. Explain which variable is now controlled and
which is allowed to change; keep both experiments rather than comparing them
under one ambiguous label.

## 6. Interpret failures and solutions

- Output sums to one but path tests fail: look for a lost terminal branch.
- Rare-prefix test fails: condition before exponentiation.
- KL is NaN: inspect zero targets and log-softmax.
- Lower training loss but worse held-out behavior: inspect overfitting, target
  quality, and objective differences; do not assume conversion is at fault.
- More units/second but slower requests: compare the same amount of raw content.

For AI-assisted coding, write the four-token example and one failing invariant
yourself. Ask the assistant for a single function, run the grader, then explain
each failure before requesting another edit. Deliberately remove EOT once and
predict which checks break. Do not let the assistant rewrite the grader to accept
its implementation. The worked solution is a comparison point, not a substitute
for reconstructing the calculation.

**Exit check:** reconstruct all four token probabilities, explain why EOT is
not EOS, and defend which experimental comparisons are actually controlled.

## Research context

[Marathe et al., Breaking the Token Ceiling (2026), sections 2 and F](https://arxiv.org/abs/2609.12303)
introduces Marginalize-It and End-Of-Token conversion in a large-scale study.
Our examples, datasets, and CPU training experiment are teaching constructions,
not a reproduction of the paper. Continue with **Does the Byte Student Actually
Win?** before interpreting scaling or deployment claims.


## Work first, then compare

CPU runtime is sufficient for the exercises. NumPy and PyTorch are available in Colab. Run the starter and grading cells after implementing the TODOs. An untouched starter is expected to fail. The reference solution below replaces the functions only when you deliberately run that cell. Then rerun the checks. No account, key, model download, or paid GPU is required.


In [ ]:
"""Token-to-byte targets. CPU NumPy; no model download required.

Vocabulary entries are unique, nonempty bytes objects, not Unicode characters.
EOT=256 is a separate symbol, not a UTF-8 byte or end-of-sequence marker.
"""
import numpy as np
EOT = 256

def byte_targets(logits, vocabulary, prefix=b'', use_eot=False):
    """Return a float vector of length 256 (approximate) or 257 (with EOT).

    Condition on vocabulary entries starting with prefix. Entries ending at
    prefix contribute to EOT when enabled; otherwise discard their mass.
    Sum probabilities by NEXT byte/symbol and normalize over eligible entries.
    Normalize eligible logits stably, even if larger logits exist elsewhere.
    logits: 1-D, one per vocabulary entry; finite or -inf (zero mass).
    Reject empty/duplicate/non-bytes entries, non-bytes prefix, bad logit shape,
    NaN/+inf, and prefixes with no positive eligible mass using ValueError.
    Do not mutate inputs. The empty prefix is valid; empty tokens are not.
    """
    raise NotImplementedError

def distillation_kl(student_logits, teacher_targets):
    """Mean forward KL(teacher || student), in nats, over rows, temperature 1.

    Both inputs have shape (positions, vocabulary_size), with nonempty axes.
    Student logits must be finite. Targets must be finite, nonnegative, and
    each row sum to 1 within atol=1e-8 (rtol=0). Zero targets contribute zero.
    Reject invalid inputs with ValueError. Use stable log-softmax, not log of
    rounded probabilities. Preserve inputs.
    """
    raise NotImplementedError


In [ ]:
import numpy as np
_vocab = [b'a', b'ab', b'ac', b'b']
_prob = np.array([.1, .4, .3, .2])
_z = np.log(_prob)
_before = _z.copy()
_first = byte_targets(_z, _vocab)
assert _first.shape == (256,)
np.testing.assert_allclose(_first[[97, 98]], [.8, .2])
_approx = byte_targets(_z, _vocab, b'a')
_exact = byte_targets(_z, _vocab, b'a', True)
assert _exact.shape == (257,)
np.testing.assert_allclose(_approx[[98, 99]], [4/7, 3/7])
np.testing.assert_allclose(_exact[[98, 99, 256]], [.5, .375, .125])
for _out in [_first, _approx, _exact]:
    assert np.isfinite(_out).all() and (_out >= 0).all()
    np.testing.assert_allclose(_out.sum(), 1)
np.testing.assert_array_equal(_z, _before)

# Reconstruct each complete token path, including its final EOT probability.
_rng = np.random.default_rng(8)
for _tokens in [_vocab, [b'x', b'xy', b'xyz', b'xz'], [b'\xc3', b'\xc3\xa9', b'\xff']]:
    for _ in range(5):
        _p = _rng.dirichlet(np.ones(len(_tokens)))
        for _i, _token in enumerate(_tokens):
            _path = 1.
            for _j, _byte in enumerate(_token):
                _path *= byte_targets(np.log(_p), _tokens, _token[:_j], True)[_byte]
            _path *= byte_targets(np.log(_p), _tokens, _token, True)[256]
            np.testing.assert_allclose(_path, _p[_i], atol=1e-12, rtol=1e-10)
np.testing.assert_allclose(byte_targets([10000, 0, 0], [b'a', b'bx', b'by'], b'b')[[120, 121]], [.5, .5])
assert byte_targets([0, -np.inf], [b'a', b'ab'], b'a', True)[256] == 1

_targets = np.array([[.7, .3, 0.], [0., 0., 1.]])
_student = np.array([[10001., 10000., 9999.], [-1000., 0., 1000.]])
_saved = _student.copy()
_shift = _student - _student.max(1, keepdims=True)
_logq = _shift - np.log(np.exp(_shift).sum(1, keepdims=True))
_expected = sum(_targets[i,j] * (np.log(_targets[i,j]) - _logq[i,j])
                for i in range(2) for j in range(3) if _targets[i,j] > 0) / 2
np.testing.assert_allclose(distillation_kl(_student, _targets), _expected, atol=1e-12)
np.testing.assert_allclose(distillation_kl(np.log([[.2, .8]]), [[.2, .8]]), 0, atol=1e-12)
np.testing.assert_array_equal(_student, _saved)
for _call in [
    lambda: byte_targets([0], [b'a'], b'a'),
    lambda: byte_targets([0], [b'a'], b'z', True),
    lambda: byte_targets([-np.inf], [b'a']),
    lambda: byte_targets([0, 0], [b'a', b'a']),
    lambda: byte_targets([0], [b'']),
    lambda: byte_targets([0], ['a']),
    lambda: byte_targets([0], [b'a'], 'a'),
    lambda: byte_targets([np.nan], [b'a']),
    lambda: byte_targets([np.inf], [b'a']),
    lambda: byte_targets([[0]], [b'a']),
    lambda: byte_targets([0, 1], [b'a']),
    lambda: distillation_kl([[0, 0]], [[.2, .2]]),
    lambda: distillation_kl([[0, 0]], [[-1, 2]]),
    lambda: distillation_kl([[np.inf, 0]], [[0, 1]]),
    lambda: distillation_kl([[0, 0]], [[np.nan, 1]]),
    lambda: distillation_kl([], []),
]:
    try:
        _call()
    except ValueError:
        pass
    else:
        raise AssertionError('invalid or unsupported input must raise ValueError')
print('Byte targets conserve path probability; approximate boundary loss and stable KL verified.')


## Worked reference solution

Compare the state transitions and invariants with your implementation. Passing output tests alone does not establish memory efficiency or reproduce the paper's performance results.


In [ ]:
import numpy as np
EOT = 256

def byte_targets(logits, vocabulary, prefix=b'', use_eot=False):
    z = np.asarray(logits, dtype=np.float64)
    if (z.ndim != 1 or len(z) != len(vocabulary) or not len(z)
            or np.isnan(z).any() or np.isposinf(z).any()
            or not isinstance(prefix, bytes)
            or any(not isinstance(t, bytes) or not t for t in vocabulary)):
        raise ValueError('invalid logits, vocabulary, or prefix')
    if len(set(vocabulary)) != len(vocabulary):
        raise ValueError('vocabulary byte strings must be unique')
    indices, symbols = [], []
    for i, token in enumerate(vocabulary):
        if token.startswith(prefix):
            if len(token) > len(prefix):
                indices.append(i)
                symbols.append(token[len(prefix)])
            elif use_eot:
                indices.append(i)
                symbols.append(EOT)
    if not indices or not np.isfinite(z[indices]).any():
        raise ValueError('prefix has no positive eligible probability')
    # Condition BEFORE exponentiation so an unlikely prefix does not underflow.
    selected = z[indices]
    weights = np.exp(selected - selected.max())
    weights /= weights.sum()
    return np.bincount(symbols, weights=weights, minlength=257 if use_eot else 256)

def distillation_kl(student_logits, teacher_targets):
    z, p = np.asarray(student_logits, dtype=np.float64), np.asarray(teacher_targets, dtype=np.float64)
    if (z.ndim != 2 or z.shape != p.shape or min(z.shape) == 0
            or not np.isfinite(z).all() or not np.isfinite(p).all()
            or (p < 0).any() or not np.allclose(p.sum(1), 1, atol=1e-8, rtol=0)):
        raise ValueError('expected finite logits and normalized target rows')
    shifted = z - z.max(1, keepdims=True)
    log_q = shifted - np.log(np.exp(shifted).sum(1, keepdims=True))
    log_p = np.log(np.where(p > 0, p, 1))
    return float(np.sum(p * (log_p - log_q), axis=1).mean())


In [ ]:
import numpy as np
_vocab = [b'a', b'ab', b'ac', b'b']
_prob = np.array([.1, .4, .3, .2])
_z = np.log(_prob)
_before = _z.copy()
_first = byte_targets(_z, _vocab)
assert _first.shape == (256,)
np.testing.assert_allclose(_first[[97, 98]], [.8, .2])
_approx = byte_targets(_z, _vocab, b'a')
_exact = byte_targets(_z, _vocab, b'a', True)
assert _exact.shape == (257,)
np.testing.assert_allclose(_approx[[98, 99]], [4/7, 3/7])
np.testing.assert_allclose(_exact[[98, 99, 256]], [.5, .375, .125])
for _out in [_first, _approx, _exact]:
    assert np.isfinite(_out).all() and (_out >= 0).all()
    np.testing.assert_allclose(_out.sum(), 1)
np.testing.assert_array_equal(_z, _before)

# Reconstruct each complete token path, including its final EOT probability.
_rng = np.random.default_rng(8)
for _tokens in [_vocab, [b'x', b'xy', b'xyz', b'xz'], [b'\xc3', b'\xc3\xa9', b'\xff']]:
    for _ in range(5):
        _p = _rng.dirichlet(np.ones(len(_tokens)))
        for _i, _token in enumerate(_tokens):
            _path = 1.
            for _j, _byte in enumerate(_token):
                _path *= byte_targets(np.log(_p), _tokens, _token[:_j], True)[_byte]
            _path *= byte_targets(np.log(_p), _tokens, _token, True)[256]
            np.testing.assert_allclose(_path, _p[_i], atol=1e-12, rtol=1e-10)
np.testing.assert_allclose(byte_targets([10000, 0, 0], [b'a', b'bx', b'by'], b'b')[[120, 121]], [.5, .5])
assert byte_targets([0, -np.inf], [b'a', b'ab'], b'a', True)[256] == 1

_targets = np.array([[.7, .3, 0.], [0., 0., 1.]])
_student = np.array([[10001., 10000., 9999.], [-1000., 0., 1000.]])
_saved = _student.copy()
_shift = _student - _student.max(1, keepdims=True)
_logq = _shift - np.log(np.exp(_shift).sum(1, keepdims=True))
_expected = sum(_targets[i,j] * (np.log(_targets[i,j]) - _logq[i,j])
                for i in range(2) for j in range(3) if _targets[i,j] > 0) / 2
np.testing.assert_allclose(distillation_kl(_student, _targets), _expected, atol=1e-12)
np.testing.assert_allclose(distillation_kl(np.log([[.2, .8]]), [[.2, .8]]), 0, atol=1e-12)
np.testing.assert_array_equal(_student, _saved)
for _call in [
    lambda: byte_targets([0], [b'a'], b'a'),
    lambda: byte_targets([0], [b'a'], b'z', True),
    lambda: byte_targets([-np.inf], [b'a']),
    lambda: byte_targets([0, 0], [b'a', b'a']),
    lambda: byte_targets([0], [b'']),
    lambda: byte_targets([0], ['a']),
    lambda: byte_targets([0], [b'a'], 'a'),
    lambda: byte_targets([np.nan], [b'a']),
    lambda: byte_targets([np.inf], [b'a']),
    lambda: byte_targets([[0]], [b'a']),
    lambda: byte_targets([0, 1], [b'a']),
    lambda: distillation_kl([[0, 0]], [[.2, .2]]),
    lambda: distillation_kl([[0, 0]], [[-1, 2]]),
    lambda: distillation_kl([[np.inf, 0]], [[0, 1]]),
    lambda: distillation_kl([[0, 0]], [[np.nan, 1]]),
    lambda: distillation_kl([], []),
]:
    try:
        _call()
    except ValueError:
        pass
    else:
        raise AssertionError('invalid or unsupported input must raise ValueError')
print('Byte targets conserve path probability; approximate boundary loss and stable KL verified.')


## Experiment and interpretation

Predict the result before running. Change one variable, repeat, and record what changed. Separate measured outcomes from assumptions.


In [ ]:
"""A learned token teacher and four byte-student arms on synthetic segmented text.

CPU: uv run --with numpy --with torch python experiments/byte_student.py
No pretrained weights, private data, or GPU. This tests a mechanism, not a
scaling frontier. The same raw-data schedule is NOT equal compute across arms.
"""
import json
import math
import time
from pathlib import Path
import runpy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB = [b'a', b'ab', b'ac', b'b', b'ba', b'c', b'ca']
BYTE_BOS, BYTE_EOT = 257, 256

class ByteLabLM(nn.Module):
    def __init__(self, inputs, outputs, width, context):
        super().__init__()
        self.embedding = nn.Embedding(inputs, width)
        self.position = nn.Parameter(torch.zeros(context, width))
        self.norm1, self.norm2 = nn.LayerNorm(width), nn.LayerNorm(width)
        self.qkv, self.proj = nn.Linear(width, 3*width), nn.Linear(width, width)
        self.ff = nn.Sequential(nn.Linear(width, 4*width), nn.GELU(), nn.Linear(4*width, width))
        self.head = nn.Linear(width, outputs)

    def forward(self, ids):
        x = self.embedding(ids) + self.position[:ids.shape[1]]
        q, k, v = self.qkv(self.norm1(x)).chunk(3, dim=-1)
        x = x + self.proj(F.scaled_dot_product_attention(q, k, v, is_causal=True))
        return self.head(x + self.ff(self.norm2(x)))

def segmented_data(count, seed, excluded=()):
    rng = np.random.default_rng(seed)
    rows, seen = [], set(excluded)
    seen_text = {b''.join(VOCAB[i] for i in row) for row in seen}
    while len(rows) < count:
        row = [int(rng.integers(len(VOCAB)))]
        for _ in range(7):
            row.append(int((row[-1] + rng.choice([1, 2, 3], p=[.65, .25, .1])) % len(VOCAB)))
        row = tuple(row)
        text = b''.join(VOCAB[i] for i in row)
        if row not in seen and text not in seen_text:
            rows.append(row)
            seen.add(row)
            seen_text.add(text)
    return torch.tensor(rows)

def token_inputs(ids):
    return torch.cat([torch.full((len(ids), 1), len(VOCAB)), ids[:, :-1]], dim=1)

def byte_dataset(ids, teacher_logits, with_eot, convert):
    size = 257 if with_eot else 256
    sequences, targets, content = [], [], []
    for sequence, logits in zip(ids.tolist(), teacher_logits.numpy()):
        symbols, soft = [], []
        for token_id, teacher_row in zip(sequence, logits):
            token = VOCAB[token_id]
            for offset, symbol in enumerate(list(token) + ([BYTE_EOT] if with_eot else [])):
                symbols.append(symbol)
                soft.append(convert(teacher_row, VOCAB, token[:offset], with_eot))
        sequences.append(symbols)
        targets.append(np.array(soft))
        content.append(sum(len(VOCAB[i]) for i in sequence))
    length = max(map(len, sequences))
    x = torch.full((len(ids), length), BYTE_BOS)
    y = torch.zeros_like(x)
    p = torch.zeros(len(ids), length, size)
    mask = torch.zeros(len(ids), length, dtype=torch.bool)
    for row, (symbols, soft) in enumerate(zip(sequences, targets)):
        n = len(symbols)
        x[row, :n] = torch.tensor([BYTE_BOS] + symbols[:-1])
        y[row, :n] = torch.tensor(symbols)
        p[row, :n] = torch.from_numpy(soft).float()
        mask[row, :n] = True
    assert torch.allclose(p.sum(-1)[mask], torch.ones_like(p.sum(-1)[mask]), atol=1e-6)
    return x, y, p, mask, torch.tensor(content)

def student_loss(logits, y, targets, mask, distilled):
    log_q = logits.log_softmax(-1)
    if distilled:
        losses = (targets * (targets.clamp_min(1e-30).log() - log_q)).sum(-1)
    else:
        losses = -log_q.gather(-1, y.unsqueeze(-1)).squeeze(-1)
    return losses[mask].mean()

@torch.no_grad()
def decode_fixed_content(model, mode, count=8):
    ids, output = [len(VOCAB) if mode == 'token' else BYTE_BOS], bytearray()
    for _ in range(64):
        logits = model(torch.tensor([ids]))[0, -1]
        if mode != 'token':
            # Declared toy constrained decoding: ASCII a/b/c, plus EOT if present.
            allowed = [97, 98, 99] + ([BYTE_EOT] if mode == 'eot' else [])
            blocked = torch.full_like(logits, -torch.inf)
            blocked[allowed] = logits[allowed]
            logits = blocked
        choice = int(logits.argmax())
        ids.append(choice)
        if mode == 'token':
            output.extend(VOCAB[choice])
        elif choice != BYTE_EOT:
            output.append(choice)
        if len(output) >= count:
            return bytes(output[:count]), len(ids)-1
    return bytes(output), len(ids)-1

def latency_report(model, mode):
    model.eval()
    for _ in range(2):
        decode_fixed_content(model, mode)
    samples, complete = [], True
    for _ in range(5):
        start = time.perf_counter()
        output, units = decode_fixed_content(model, mode)
        samples.append((time.perf_counter()-start)*1000)
        complete = complete and len(output) == 8
    return {'completed_8_bytes': complete, 'generated_ascii': output.decode('ascii'),
            'last_decode_units': units, 'median_ms': float(np.median(samples)),
            'p95_ms_nearest_rank': max(samples),
            'sequential_bytes_per_second': 40000/sum(samples) if complete else None,
            'parameter_storage_bytes': sum(p.numel()*p.element_size() for p in model.parameters())}

def run_byte_experiment(convert=None, steps=50, seeds=(3, 13)):
    if convert is None:
        file = Path(__file__).resolve().parents[1] / 'public/content/solutions/c9/09_byte_distillation.py'
        convert = runpy.run_path(str(file))['byte_targets']
    torch.set_num_threads(1)
    torch.manual_seed(23)
    train = segmented_data(192, 101)
    val = segmented_data(64, 202, map(tuple, train.tolist()))
    teacher = ByteLabLM(8, 7, 48, 9)
    optimizer = torch.optim.AdamW(teacher.parameters(), lr=.006)
    for step in range(80):
        idx = torch.arange(step*32, (step+1)*32) % len(train)
        loss = F.cross_entropy(teacher(token_inputs(train[idx])).flatten(0,1), train[idx].flatten())
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    teacher.eval()
    with torch.no_grad():
        train_logits, val_logits = teacher(token_inputs(train)), teacher(token_inputs(val))
        teacher_nll = F.cross_entropy(val_logits.flatten(0,1), val.flatten(), reduction='sum')
    datasets = {eot: (byte_dataset(train, train_logits, eot, convert), byte_dataset(val, val_logits, eot, convert)) for eot in [False, True]}
    teacher_params = sum(p.numel() for p in teacher.parameters())
    rows = []
    for seed in seeds:
        for eot, distilled in [(False, False), (False, True), (True, False), (True, True)]:
            torch.manual_seed(seed)
            model = ByteLabLM(258, 257 if eot else 256, 16, 65)
            parameters = sum(p.numel() for p in model.parameters())
            assert parameters < teacher_params
            optimizer = torch.optim.AdamW(model.parameters(), lr=.01)
            (x, y, p, mask, content), validation = datasets[eot]
            initial = float(student_loss(model(x[:16]), y[:16], p[:16], mask[:16], distilled).detach())
            symbols_seen, content_seen = 0, 0
            started = time.perf_counter()
            for step in range(steps):
                idx = torch.arange(step*16, (step+1)*16) % len(x)
                for group in optimizer.param_groups:
                    group['lr'] = .01 * (.1 + .9 * .5 * (1 + math.cos(math.pi*step/steps)))
                loss = student_loss(model(x[idx]), y[idx], p[idx], mask[idx], distilled)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                symbols_seen += int(mask[idx].sum())
                content_seen += int(content[idx].sum())
            training_seconds = time.perf_counter()-started
            model.eval()
            with torch.no_grad():
                final = float(student_loss(model(x[:16]), y[:16], p[:16], mask[:16], distilled))
                vx, vy, _, vm, vc = validation
                nll = F.cross_entropy(model(vx).flatten(0,1), vy.flatten(), reduction='none').reshape_as(vy)
                bpb = float(nll[vm].sum() / (vc.sum()*math.log(2)))
            assert math.isfinite(bpb) and math.isfinite(final)
            rows.append({'seed': seed, 'representation': 'bytes+EOT' if eot else 'bytes',
                         'objective': 'distillation_KL' if distilled else 'supervised_CE',
                         'parameters': parameters, 'train_content_bytes_seen': content_seen,
                         'train_prediction_units_seen': symbols_seen,
                         'parameter_unit_work_proxy_6NU': 6*parameters*symbols_seen,
                         'training_seconds': training_seconds, 'initial_train_loss': initial,
                         'final_train_loss': final, 'validation_representation_BPB': bpb,
                         'serving': latency_report(model, 'eot' if eot else 'bytes')})
    return {'setup': {'torch': torch.__version__, 'device': 'CPU', 'threads': 1,
                      'teacher_parameters': teacher_params, 'teacher_training_steps': 80,
                      'student_steps': steps, 'teacher_representation_BPB': float(teacher_nll/(datasets[False][1][4].sum()*math.log(2))),
                      'teacher_serving': latency_report(teacher, 'token')}, 'rows': rows}

if __name__ == '__main__':
    RESULTS = run_byte_experiment(globals().get('byte_targets'))
    print(json.dumps(RESULTS, indent=2))
    print('Synthetic segmented text, not a learned BPE tokenizer or the paper corpus. Fixed raw-data schedule, unequal compute. 6NU is a work proxy, not measured FLOPs. Teacher training/conversion costs are separate. BPB scores the declared representation, including EOT losses where present. CPU decoding is constrained to a/b/c and 8 returned bytes, with no KV cache; tensor storage is not peak RAM. No scaling or production-quality win is established.')
